# Protótipo de automação com IA aplicada à gestão de contratos

**TCC de Engenharia de Produção — Mapeamento do processo de gestão de contratos e proposta de automação com IA: uma abordagem sob a ótica do Lean Office**

Este notebook implementa e valida o protótipo descrito na Fundamentação Teórica e na Metodologia do trabalho, seguindo a arquitetura de 3 módulos:

1. **Extração de texto** dos contratos (PDF → texto ou, quando necessário, imagem);
2. **Extração de campos-chave via LLM** (modelo Claude, API da Anthropic);
3. **Comparação com o gabarito** (ground-truth extraído manualmente) e cálculo de indicadores de acerto.

A amostra utilizada é composta por 8 contratos públicos reais, obtidos no Portal Nacional de Contratações Públicas (PNCP), conforme os critérios de seleção descritos no capítulo de Materiais e Métodos.

**Pré-requisito para rodar este notebook:** um arquivo `.env` na raiz do projeto (`tcc_prototipo/.env`) contendo `ANTHROPIC_API_KEY=<sua chave>`. A chave nunca é exibida nem gravada nas saídas deste notebook.

## Módulo 1 — Extração de texto dos contratos

Módulo 1 - Extração de texto
Protótipo de automação com IA aplicada à gestão de contratos

Este script implementa a primeira etapa da arquitetura de 3 módulos descrita
na Fundamentação Teórica: extração do texto bruto dos contratos em PDF,
para posterior processamento pelo módulo de extração de campos-chave (via LLM).

In [ ]:
import pdfplumber
from pathlib import Path
import json

CONTRATOS_DIR = Path.cwd() / "contratos"
TEXTOS_DIR = Path.cwd() / "contratos" / "_texto_extraido" 
TEXTOS_DIR.mkdir(exist_ok=True)


def extrair_texto_pdf(caminho_pdf: Path) -> str:
    """Extrai todo o texto de um PDF, página por página, usando pdfplumber."""
    texto_paginas = []
    with pdfplumber.open(caminho_pdf) as pdf:
        for i, pagina in enumerate(pdf.pages, start=1):
            texto = pagina.extract_text() or ""
            texto_paginas.append(f"\n--- PÁGINA {i} ---\n{texto}")
    return "\n".join(texto_paginas)


def main():
    resultados = {}
    pdfs = sorted(CONTRATOS_DIR.glob("*.pdf"))
    print(f"Encontrados {len(pdfs)} contratos para processar.\n")

    for pdf_path in pdfs:
        nome = pdf_path.stem
        print(f"Extraindo texto: {nome} ...")
        texto = extrair_texto_pdf(pdf_path)
        n_paginas = texto.count("--- PÁGINA")
        n_chars = len(texto)

        # salva o texto extraído em .txt para inspeção/uso posterior
        txt_path = TEXTOS_DIR / f"{nome}.txt"
        txt_path.write_text(texto, encoding="utf-8")

        resultados[nome] = {
            "arquivo_pdf": pdf_path.name,
            "n_paginas": n_paginas,
            "n_caracteres": n_chars,
            "arquivo_texto": str(txt_path.relative_to(CONTRATOS_DIR.parent)),
        }
        print(f"  -> {n_paginas} páginas, {n_chars} caracteres extraídos.\n")

    # salva um resumo em JSON (útil para o capítulo de Materiais e Métodos)
    resumo_path = CONTRATOS_DIR.parent / "gabarito" / "resumo_extracao_texto.json"
    resumo_path.write_text(
        json.dumps(resultados, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Resumo salvo em: {resumo_path}")


main()

## Módulo 1 (complemento) — Contratos digitalizados sem texto nativo

Módulo 1 (complemento) - Tratamento de contratos digitalizados sem camada de texto

Dois dos 8 contratos da amostra (PRF/General Motors e PM-RJ/CS Brasil) são
digitalizações (scans) sem camada de texto pesquisável.

Em vez de montar um pipeline de OCR dedicado (o pacote de idioma português do
Tesseract não pôde ser instalado neste ambiente por restrição de rede), a
solução adotada foi renderizar cada página em imagem e enviá-la diretamente
ao módulo de extração via LLM multimodal (que lê imagem + texto), evitando
uma etapa intermediária de OCR e sua perda de acurácia em textos jurídicos
com muitos números e cláusulas numeradas.

In [ ]:
import pypdfium2 as pdfium
from pathlib import Path
import json

CONTRATOS_DIR = Path.cwd() / "contratos"  
IMG_DIR = CONTRATOS_DIR / "_paginas_imagem"
IMG_DIR.mkdir(exist_ok=True)

# contratos identificados com camada de texto insuficiente
CONTRATOS_ESCANEADOS = [
    "3_PRF-GM_21-2024",
    "6_PMRJ-CSBrasil_018-2025",
]

DPI_RENDER = 150  # resolução suficiente para leitura de texto pequeno sem ficar pesado


def renderiza_pdf_em_imagens(nome_contrato: str) -> list[str]:
    pdf_path = CONTRATOS_DIR / f"{nome_contrato}.pdf"
    pdf = pdfium.PdfDocument(str(pdf_path))
    caminhos = []
    escala = DPI_RENDER / 72  # pdfium usa escala relativa a 72 DPI

    out_subdir = IMG_DIR / nome_contrato
    out_subdir.mkdir(exist_ok=True)

    for i in range(len(pdf)):
        pagina = pdf[i]
        bitmap = pagina.render(scale=escala)
        pil_image = bitmap.to_pil()
        out_path = out_subdir / f"pagina_{i+1:02d}.png"
        pil_image.save(out_path)
        caminhos.append(str(out_path.relative_to(CONTRATOS_DIR.parent)))

    return caminhos


def main():
    resultado = {}
    for nome in CONTRATOS_ESCANEADOS:
        print(f"Renderizando páginas de: {nome} ...")
        caminhos = renderiza_pdf_em_imagens(nome)
        resultado[nome] = {
            "n_paginas_renderizadas": len(caminhos),
            "dpi": DPI_RENDER,
            "caminhos": caminhos,
        }
        print(f"  -> {len(caminhos)} páginas renderizadas em {IMG_DIR / nome}\n")

    resumo_path = CONTRATOS_DIR.parent / "gabarito" / "resumo_paginas_imagem.json"
    resumo_path.write_text(
        json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Resumo salvo em: {resumo_path}")


main()

## Módulo 2 — Extração de campos-chave via LLM

Módulo 2 - Extração de campos-chave via LLM


Envia cada contrato (texto nativo, ou imagens das páginas quando o PDF é uma
digitalização sem camada de texto) para a API da Anthropic (Claude), pedindo
a extração estruturada dos mesmos 14 campos definidos no gabarito manual.

Modelo utilizado: claude-haiku-4-5 (economia de custo/tempo, adequado a uma
tarefa de extração estruturada de campos, sem necessidade de raciocínio
complexo). Parâmetro de temperatura fixado em 0 para maximizar
determinismo/reprodutibilidade da extração (não é uma tarefa criativa).

In [ ]:
import base64
import json
import time
from pathlib import Path

import requests

BASE_DIR = Path.cwd() 
ENV_PATH = BASE_DIR / ".env"
CONTRATOS_DIR = BASE_DIR / "contratos"
TEXTOS_DIR = CONTRATOS_DIR / "_texto_extraido"
IMAGENS_DIR = CONTRATOS_DIR / "_paginas_imagem"
GABARITO_PATH = BASE_DIR / "gabarito" / "gabarito.json"
RESULTADOS_DIR = BASE_DIR / "gabarito" / "resultados_llm"
RESULTADOS_DIR.mkdir(exist_ok=True)

MODEL = "claude-haiku-4-5-20251001"
API_URL = "https://api.anthropic.com/v1/messages"
API_VERSION = "2023-06-01"

# contratos cuja extração precisa usar imagem das páginas (sem texto nativo)
CONTRATOS_IMAGEM = {"3_TCESP-GM_21-2024", "6_PMRJ-CSBrasil_018-2025"}
# nomes de arquivo na pasta de imagens ainda usam o rótulo antigo do contrato 3
MAPA_PASTA_IMAGEM = {
    "3_TCESP-GM_21-2024": "3_PRF-GM_21-2024",
    "6_PMRJ-CSBrasil_018-2025": "6_PMRJ-CSBrasil_018-2025",
}


def carrega_api_key() -> str:
    for linha in ENV_PATH.read_text().splitlines():
        if linha.startswith("ANTHROPIC_API_KEY="):
            return linha.split("=", 1)[1].strip()
    raise RuntimeError("ANTHROPIC_API_KEY não encontrada em .env")


PROMPT_SISTEMA = """Você é um módulo de extração de dados de contratos administrativos \
públicos brasileiros, regidos pela Lei nº 14.133/2021. Sua tarefa é ler o contrato \
fornecido (texto ou imagens das páginas) e extrair os campos abaixo, retornando \
ESTRITAMENTE um objeto JSON válido, sem nenhum texto antes ou depois.

Campos a extrair:
- tipo_contratacao: descrição curta do objeto/tipo de contratação
- orgao: nome do órgão/entidade contratante
- cnpj_orgao: CNPJ do órgão contratante (ou null se não encontrado)
- contratada: razão social da empresa contratada
- cnpj_contratada: CNPJ da contratada (ou null se não encontrado)
- valor: valor do contrato, como aparece no texto (com moeda e periodicidade se houver)
- vigencia_inicial_meses: duração inicial da vigência, em número de meses (inteiro)
- mecanismo_prorrogacao: como funciona a prorrogação/renovação do contrato (resuma a regra; se não houver cláusula de prorrogação, diga explicitamente "não há cláusula de prorrogação")
- garantia_execucao: percentual ou valor de garantia exigido (ou "não há cláusula" se não encontrado)
- multa_teto: teto/faixa de multa por infração (ou "não há cláusula" se não encontrado)
- gestor_fiscal: quem é designado gestor/fiscal do contrato, ou como é designado (ou "não identificado" se não encontrado)
- clausula_lgpd: resuma se há cláusula de LGPD e seu nível de detalhe, ou "sem cláusula dedicada" se não houver

IMPORTANTE: se um campo não estiver presente no documento, retorne explicitamente \
que não há essa cláusula/informação. NUNCA invente ou infira um valor que não está \
no texto. É preferível responder "não encontrado" a arriscar uma alucinação.
"""


def monta_mensagem_texto(nome_contrato: str, texto: str) -> list:
    return [
        {
            "role": "user",
            "content": f"Contrato: {nome_contrato}\n\nTEXTO DO CONTRATO:\n{texto}",
        }
    ]


def monta_mensagem_imagens(nome_contrato: str, caminhos_imagens: list) -> list:
    blocos = [{"type": "text", "text": f"Contrato: {nome_contrato}\n\nSegue abaixo as páginas do contrato, como imagens, em ordem."}]
    for caminho in caminhos_imagens:
        img_bytes = (BASE_DIR / caminho).read_bytes()
        b64 = base64.standard_b64encode(img_bytes).decode("utf-8")
        blocos.append(
            {
                "type": "image",
                "source": {"type": "base64", "media_type": "image/png", "data": b64},
            }
        )
    return [{"role": "user", "content": blocos}]


def chama_api(api_key: str, mensagens: list) -> tuple[dict, float]:
    inicio = time.time()
    resp = requests.post(
        API_URL,
        headers={
            "x-api-key": api_key,
            "anthropic-version": API_VERSION,
            "content-type": "application/json",
        },
        json={
            "model": MODEL,
            "max_tokens": 1500,
            "temperature": 0,
            "system": PROMPT_SISTEMA,
            "messages": mensagens,
        },
        timeout=120,
    )
    duracao = time.time() - inicio
    resp.raise_for_status()
    return resp.json(), duracao


def extrai_json_da_resposta(resposta: dict) -> dict:
    texto = "".join(bloco.get("text", "") for bloco in resposta.get("content", []))
    texto = texto.strip()
    # remove eventuais cercas de markdown ```json ... ```
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    return json.loads(texto.strip())


def main():
    api_key = carrega_api_key()
    gabarito = json.loads(GABARITO_PATH.read_text(encoding="utf-8"))
    ids_contratos = [c["id"] for c in gabarito["contratos"]]

    log_tempos = {}

    for nome in ids_contratos:
        print(f"Processando: {nome} ...")

        if nome in CONTRATOS_IMAGEM:
            pasta_imagem = MAPA_PASTA_IMAGEM[nome]
            caminhos = sorted((IMAGENS_DIR / pasta_imagem).glob("*.png"))
            caminhos_rel = [str(c.relative_to(BASE_DIR)) for c in caminhos]
            mensagens = monta_mensagem_imagens(nome, caminhos_rel)
            modo = "imagem"
        else:
            texto = (TEXTOS_DIR / f"{nome}.txt").read_text(encoding="utf-8")
            mensagens = monta_mensagem_texto(nome, texto)
            modo = "texto"

        try:
            resposta_bruta, duracao = chama_api(api_key, mensagens)
            campos_extraidos = extrai_json_da_resposta(resposta_bruta)
            uso = resposta_bruta.get("usage", {})
            status = "ok"
        except Exception as e:
            campos_extraidos = {"erro": str(e)}
            duracao = None
            uso = {}
            status = "erro"

        log_tempos[nome] = {
            "modo_entrada": modo,
            "duracao_segundos": round(duracao, 2) if duracao else None,
            "tokens_entrada": uso.get("input_tokens"),
            "tokens_saida": uso.get("output_tokens"),
            "status": status,
        }

        out_path = RESULTADOS_DIR / f"{nome}.json"
        out_path.write_text(
            json.dumps(campos_extraidos, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print(f"  -> modo={modo}, status={status}, tempo={log_tempos[nome]['duracao_segundos']}s, "
              f"tokens_in={uso.get('input_tokens')}, tokens_out={uso.get('output_tokens')}")

    log_path = BASE_DIR / "gabarito" / "log_processamento.json"
    log_path.write_text(json.dumps(log_tempos, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nLog de processamento salvo em: {log_path}")


main()

## Módulo 3 — Comparação com o gabarito e cálculo de indicadores

Módulo 3 - Comparação com o gabarito e cálculo de indicadores
Protótipo de automação com IA aplicada à gestão de contratos


Compara a saída do LLM (módulo 2) com o gabarito (ground-truth extraído
manualmente) e calcula indicadores de precisão de extração, separando:
- Campos "estruturados" (valor, CNPJ, meses de vigência etc.): comparação
  automática por presença dos mesmos números/tokens-chave.
- Campos "descritivos" (mecanismo de prorrogação, LGPD, multa, gestor):
  sinalizados para revisão humana, com uma checagem automática auxiliar
  (concordância quanto à PRESENÇA ou AUSÊNCIA da cláusula), já que a
  redação livre não permite comparação exata palavra por palavra.

In [ ]:
import json
import re
from pathlib import Path

BASE_DIR = Path.cwd()  
GABARITO_PATH = BASE_DIR / "gabarito" / "gabarito.json"
RESULTADOS_DIR = BASE_DIR / "gabarito" / "resultados_llm"

CAMPOS_ESTRUTURADOS = ["cnpj_orgao", "cnpj_contratada", "vigencia_inicial_meses"]
CAMPOS_DESCRITIVOS = [
    "tipo_contratacao", "orgao", "contratada", "valor",
    "mecanismo_prorrogacao", "garantia_execucao", "multa_teto",
    "gestor_fiscal", "clausula_lgpd",
]

FRASES_AUSENCIA = [
    "não há", "nao ha", "sem cláusula", "sem clausula", "não identificado", "nao identificado",
    "não encontrado", "nenhuma", "nenhum", "não exigid", "nao exigid", "não nomeado", "nao nomeado",
    "não consta", "nao consta", "sem exigência", "sem exigencia", "sem garantia", "não possui",
    "nao possui", "não é exigida", "nao e exigida", "não tem", "nao tem", "sem cobrança", "não requer",
]


def normaliza(valor) -> str:
    if valor is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())


def indica_ausencia(texto: str) -> bool:
    if not texto:
        return True
    texto_lower = str(texto).lower()
    return any(frase in texto_lower for frase in FRASES_AUSENCIA)


def compara_campo_estruturado(esperado, obtido) -> str:
    if esperado is None and obtido is None:
        return "N/A"
    n_esperado = normaliza(esperado)
    n_obtido = normaliza(obtido)
    if not n_esperado:
        return "N/A"
    if n_esperado == n_obtido:
        return "OK"
    # para CNPJ, checa se os dígitos batem mesmo com formatação diferente
    if n_esperado in n_obtido or n_obtido in n_esperado:
        return "OK (parcial)"
    return "DIVERGENTE"


def compara_campo_descritivo(esperado, obtido) -> str:
    ausencia_esperada = indica_ausencia(esperado)
    ausencia_obtida = indica_ausencia(obtido)
    if ausencia_esperada and ausencia_obtida:
        return "OK (concordam: ausente)"
    if ausencia_esperada != ausencia_obtida:
        return "DIVERGENTE (presença/ausência)"
    return "REVISAR (ambos indicam presença; conteúdo requer leitura humana)"


def main():
    gabarito = json.loads(GABARITO_PATH.read_text(encoding="utf-8"))
    linhas_relatorio = []
    contagem = {"OK": 0, "OK (parcial)": 0, "DIVERGENTE": 0, "N/A": 0}

    for contrato in gabarito["contratos"]:
        nome = contrato["id"]
        resultado_path = RESULTADOS_DIR / f"{nome}.json"
        if not resultado_path.exists():
            print(f"[AVISO] sem resultado LLM para {nome}")
            continue
        obtido = json.loads(resultado_path.read_text(encoding="utf-8"))

        print(f"\n{'='*80}\n{nome}\n{'='*80}")

        for campo in CAMPOS_ESTRUTURADOS:
            veredito = compara_campo_estruturado(contrato.get(campo), obtido.get(campo))
            contagem[veredito] = contagem.get(veredito, 0) + 1
            print(f"  [{veredito:14s}] {campo}: esperado={contrato.get(campo)!r} | obtido={obtido.get(campo)!r}")
            linhas_relatorio.append({
                "contrato": nome, "campo": campo, "tipo": "estruturado",
                "esperado": contrato.get(campo), "obtido": obtido.get(campo), "veredito": veredito,
            })

        for campo in CAMPOS_DESCRITIVOS:
            veredito = compara_campo_descritivo(contrato.get(campo), obtido.get(campo))
            print(f"  [{veredito:40s}] {campo}")
            linhas_relatorio.append({
                "contrato": nome, "campo": campo, "tipo": "descritivo",
                "esperado": contrato.get(campo),
                "obtido": obtido.get(campo), "veredito": veredito,
            })

    out_path = BASE_DIR / "gabarito" / "relatorio_comparacao.json"
    out_path.write_text(json.dumps(linhas_relatorio, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"\n\nResumo de campos ESTRUTURADOS (comparação automática exata/parcial):")
    for k, v in contagem.items():
        print(f"  {k}: {v}")

    total_estruturados = sum(v for k, v in contagem.items() if k != "N/A")
    ok_estruturados = contagem.get("OK", 0) + contagem.get("OK (parcial)", 0)
    if total_estruturados:
        print(f"\nTaxa de acerto (campos estruturados): {ok_estruturados}/{total_estruturados} "
              f"= {100*ok_estruturados/total_estruturados:.1f}%")

    print(f"\nRelatório completo salvo em: {out_path}")


main()